# ARC Prize 2026 — ARC-AGI-3 Offline Submission

Prize-safe notebook for the offline controller. It does not use Anthropic, external LLM APIs, Kaggle secrets, or internet-only dependencies. The controller uses the scripted probe plus a score-delta policy from `agent/offline_controller.py`.

Before running:
1. Attach this repo as a Kaggle Dataset named `arc-prize-2026-agent`.
2. Keep Internet disabled for prize-style evaluation.
3. Click **Save & Run All**.

In [ ]:
# 1. Verify the competition runtime exposes the ARC-AGI-3 SDK.
# Do not pip install here: prize-style evaluation has internet disabled,
# and we do not import Anthropic or any other external LLM API in this notebook.
import importlib.util

if importlib.util.find_spec("arc_agi_3") is None:
    raise RuntimeError(
        "arc_agi_3 is not available in this runtime. "
        "Attach/provide the competition package offline (Python 3.12+ required)."
    )

In [ ]:
# 2. Make the agent package importable from the attached Kaggle Dataset.
import os, sys

DATASET_PATH = "/kaggle/input/arc-prize-2026-agent"
if os.path.isdir(DATASET_PATH):
    sys.path.insert(0, DATASET_PATH)
else:
    raise RuntimeError(f"{DATASET_PATH} not found. Attach the repo as a Kaggle Dataset.")

In [ ]:
# 3. Run the offline controller across every game the API exposes.
#
# arc_agi_3.Swarm only accepts a string key from AVAILABLE_AGENTS (verified via
# inspect on the installed SDK -- Swarm.__init__ does AVAILABLE_AGENTS[agent]),
# so we cannot hand it our custom OfflineControllerAgent class. Instead we
# mirror Swarm's lifecycle by hand: open a scorecard, instantiate one
# OfflineControllerAgent per game, run main(), then close the scorecard.
import json, logging, os
import requests

import arc_agi_3
from arc_agi_3 import Scorecard
from agent.offline_controller import OfflineControllerAgent

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("submission_offline")

# ROOT_URL matches the SDK CLI default (arc_agi_3._cli._get_root_url):
# scheme=https, host=three.arcprize.org. Override via ARC_ROOT_URL if needed.
ROOT_URL = os.environ.get("ARC_ROOT_URL", "https://three.arcprize.org")
AGENT_NAME = "offline-controller"
TAGS = ["offline-controller", "probe-policy"]

headers = {
    "X-API-Key": os.environ.get("ARC_API_KEY", ""),
    "Accept": "application/json",
}

session = requests.Session()
session.headers.update(headers)

# Discover games via the API (same call the SDK CLI makes).
games_resp = session.get(f"{ROOT_URL}/api/games", timeout=10)
games_resp.raise_for_status()
games = [g["game_id"] for g in games_resp.json()]
logger.info("discovered %d games: %s", len(games), games)

# Open a single scorecard shared across every per-game run.
open_resp = session.post(
    f"{ROOT_URL}/api/scorecard/open",
    json={"tags": TAGS},
    headers=headers,
)
open_resp.raise_for_status()
card_id = str(open_resp.json()["card_id"])
logger.info("opened scorecard %s", card_id)

scorecard = None
try:
    for game_id in games:
        logger.info("starting game %s", game_id)
        agent = OfflineControllerAgent(
            card_id=card_id,
            game_id=game_id,
            agent_name=AGENT_NAME,
            ROOT_URL=ROOT_URL,
            record=True,
            tags=TAGS,
            cookies=session.cookies,
        )
        agent.main()

    close_resp = session.post(
        f"{ROOT_URL}/api/scorecard/close",
        json={"card_id": card_id},
        headers=headers,
    )
    close_resp.raise_for_status()
    scorecard = Scorecard.model_validate(close_resp.json())
finally:
    session.close()

payload = scorecard.model_dump() if scorecard is not None else {"card_id": card_id, "games": games}
print(json.dumps(payload, indent=2, default=str))

In [ ]:
# 4. Persist the scorecard for Kaggle's grader.
out_path = "/kaggle/working/scorecard.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(payload, f, indent=2, default=str)
print("wrote", out_path)